# 🧠 Transformers with Keras — Sequence-to-Sequence Translation

In this notebook I build a **Transformer-style sequence-to-sequence model** using Keras to translate English sentences into Spanish. The architecture combines an LSTM-based Encoder-Decoder with a custom **Self-Attention** layer — a direct ancestor of the full Transformer ("Attention Is All You Need", Vaswani et al., 2017).

> **Telecom framing 📡:** A seq2seq model is structurally identical to a **relay node** in a communication system — the encoder compresses the source signal into a compact representation (baseband encoding), and the decoder reconstructs the target signal from that representation. Self-attention is the **matched filter bank** that decides which parts of the incoming signal are most relevant at each decoding step.

## 📋 Overview

| Property | Value |
|---|---|
| **Task** | Machine translation (English → Spanish) |
| **Architecture** | Encoder-Decoder LSTM + Self-Attention |
| **Loss** | Categorical cross-entropy |
| **Optimiser (baseline)** | Adam |
| **Key technique** | Scaled dot-product attention (Q, K, V) |
| **Module** | 02 — Intro to Deep Learning & Neural Networks with Keras |

**What I cover:**
1. Prepare a small parallel corpus and apply tokenisation + padding
2. Implement a custom `SelfAttention` Keras layer from scratch
3. Build an Encoder-Decoder model with cross-attention
4. Train and plot the baseline loss curve
5. Compare **Glorot vs He** weight initialisers
6. Compare **Adam vs Adagrad** optimisers

---
## 🧩 Theory — Self-Attention

Self-attention allows every token in a sequence to **look at every other token** and decide how much weight to give it when forming its own representation. It uses three learned linear projections:

| Projection | Symbol | What it represents |
|---|---|---|
| Query | $Q$ | What this token is **looking for** |
| Key | $K$ | What this token **advertises** about itself |
| Value | $V$ | The **actual content** this token carries |

### The math 🔢

**Step 1 — Project into Q, K, V spaces:**
$$Q = X W_Q, \quad K = X W_K, \quad V = X W_V$$

**Step 2 — Compute scaled dot-product attention scores:**
$$\text{score}(Q, K) = \frac{Q K^\top}{\sqrt{d_k}}$$

where $d_k$ is the key dimension. The $\sqrt{d_k}$ scaling prevents the dot products from growing too large (which would push softmax into saturation).

**Step 3 — Normalise with softmax and apply to Values:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

> **RF analogy 📡:** The dot product $Q \cdot K^\top$ is a **matched filter correlation** — the same operation used to align a local code replica against a received CDMA chip sequence. The softmax converts raw correlation scores into mixing weights. High correlation → high weight → the decoder routes more information from that encoder position.

---
## ⚙️ Part 1 — Environment Setup

In [ ]:
# Suppress TensorFlow CPU warnings — comment out when running on GPU
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [ ]:
import numpy as np
import warnings
import matplotlib.pyplot as plt
warnings.simplefilter('ignore', FutureWarning)

from keras.models import Model
from keras.layers import Input, LSTM, Dense, Embedding, Concatenate
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Layer

print('✅ Libraries loaded')

---
## 📥 Part 2 — Data Preparation

I use a minimal parallel corpus of 5 English → Spanish sentence pairs. Despite the tiny size, it is sufficient to verify that the model learns and the loss decreases.

### Teacher forcing 🎯

During training I use **teacher forcing**: instead of feeding the decoder its own (potentially wrong) predictions, I feed it the ground-truth previous token. This stabilises early training — like training a relay with a perfect reference signal.

Concretely:
- `decoder_input_data`  = `[startseq, w1, w2, ..., w_{n-1}]`
- `decoder_output_data` = `[w1, w2, ..., w_{n-1}, endseq]`

The model learns: *given the encoder context and all previous correct tokens, predict the next token.*

In [ ]:
# 📥 Parallel corpus: English → Spanish
input_texts = [
    'Hello.',
    'How are you?',
    'I am learning machine translation.',
    'What is your name?',
    'I love programming.'
]
target_texts = [
    'Hola.',
    '¿Cómo estás?',
    'Estoy aprendiendo traducción automática.',
    '¿Cuál es tu nombre?',
    'Me encanta programar.'
]

# Wrap targets with decoder control tokens
target_texts = ['startseq ' + x + ' endseq' for x in target_texts]
print('Sample target:', target_texts[0])

In [ ]:
# 🔢 Tokenise both vocabularies independently
input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)
input_sequences = input_tokenizer.texts_to_sequences(input_texts)

output_tokenizer = Tokenizer()
output_tokenizer.fit_on_texts(target_texts)
output_sequences = output_tokenizer.texts_to_sequences(target_texts)

# +1 for index 0 (reserved for padding)
input_vocab_size  = len(input_tokenizer.word_index) + 1
output_vocab_size = len(output_tokenizer.word_index) + 1

print(f'Input vocab size:  {input_vocab_size}')
print(f'Output vocab size: {output_vocab_size}')

In [ ]:
# ➕ Pad sequences to uniform length (post-padding with zeros)
max_input_length  = max(len(seq) for seq in input_sequences)
max_output_length = max(len(seq) for seq in output_sequences)

input_sequences  = pad_sequences(input_sequences,  maxlen=max_input_length,  padding='post')
output_sequences = pad_sequences(output_sequences, maxlen=max_output_length, padding='post')

print(f'Encoder input shape:  {input_sequences.shape}  (samples × max_input_len)')
print(f'Decoder output shape: {output_sequences.shape} (samples × max_output_len)')

In [ ]:
# 🔄 Build teacher-forcing targets
decoder_input_data  = output_sequences[:, :-1]   # all but last
decoder_output_data = output_sequences[:,  1:]   # all but first

# One-hot encode outputs (required for categorical_crossentropy)
decoder_output_data = np.array([np.eye(output_vocab_size)[seq] for seq in decoder_output_data])

print(f'Decoder input  shape: {decoder_input_data.shape}')
print(f'Decoder output shape: {decoder_output_data.shape}  ← one-hot encoded')

---
## 🏗️ Part 3 — Self-Attention Layer

I implement `SelfAttention` as a custom Keras `Layer`. The three key methods are:

| Method | Purpose |
|---|---|
| `build()` | Allocate trainable weight matrices $W_Q$, $W_K$, $W_V$ of shape `(feature_dim, feature_dim)` |
| `call()` | Project inputs → compute scaled dot-product scores → softmax → weighted sum of Values |
| `compute_output_shape()` | Output shape = query shape (attention preserves dimensions) |

Weights are initialised with **Glorot uniform** (Xavier), which scales by:
$$W \sim \mathcal{U}\left(-\sqrt{\frac{6}{fan\_in + fan\_out}},\ +\sqrt{\frac{6}{fan\_in + fan\_out}}\right)$$

This balances gradient flow in both directions — the right choice for symmetric activations like `tanh` and `sigmoid`.

In [ ]:
from tensorflow.keras.layers import Layer
from tensorflow.keras import backend as K


class SelfAttention(Layer):
    """⚙️ Scaled dot-product self-attention layer.

    Accepts inputs=[query, key, value] — three tensors of shape
    (batch, seq_len, feature_dim) — and returns an attended output
    of the same shape as the query.

    Math:
        Attention(Q, K, V) = softmax( Q·Kᵀ / √dₖ ) · V
    """

    def __init__(self, **kwargs):
        super(SelfAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        # input_shape is a list: [q_shape, k_shape, v_shape]
        feature_dim = input_shape[0][-1]  # last axis = embedding dimension

        # Three square projection matrices: feature_dim → feature_dim
        self.Wq = self.add_weight(
            shape=(feature_dim, feature_dim),
            initializer='glorot_uniform',
            trainable=True, name='Wq'
        )
        self.Wk = self.add_weight(
            shape=(feature_dim, feature_dim),
            initializer='glorot_uniform',
            trainable=True, name='Wk'
        )
        self.Wv = self.add_weight(
            shape=(feature_dim, feature_dim),
            initializer='glorot_uniform',
            trainable=True, name='Wv'
        )
        super(SelfAttention, self).build(input_shape)

    def call(self, inputs):
        q, k, v = inputs

        # Project into Q, K, V spaces
        q = K.dot(q, self.Wq)  # (batch, seq_q, feature_dim)
        k = K.dot(k, self.Wk)  # (batch, seq_k, feature_dim)
        v = K.dot(v, self.Wv)  # (batch, seq_v, feature_dim)

        # Scaled dot-product: scores shape = (batch, seq_q, seq_k)
        scores = K.batch_dot(q, k, axes=[2, 2])
        dk     = K.cast(K.shape(k)[-1], dtype=K.floatx())
        scores = scores / K.sqrt(dk)          # ÷ √dₖ prevents softmax saturation

        # Attention weights → weighted sum of Values
        attention_weights = K.softmax(scores, axis=-1)  # (batch, seq_q, seq_k)
        output = K.batch_dot(attention_weights, v)       # (batch, seq_q, feature_dim)

        return output

    def compute_output_shape(self, input_shape):
        # Attention preserves query shape — no dimensional change
        return input_shape[0]


print('✅ SelfAttention layer defined')

---
## 🏗️ Part 4 — Model Architecture

The full model is an **Encoder-Decoder with Cross-Attention**:

```
English tokens
      │
  Embedding(256)              Spanish tokens
      │                             │
  LSTM(256, return_seqs=True)   Embedding(256)
  encoder_outputs ─────────┐        │
  encoder_states (h, c) ───┼──▶ LSTM(256, init_state=enc_states)
                           │   decoder_outputs
                           │        │
                           └──▶ SelfAttention(Q=dec, K=enc, V=enc)
                                attention_context
                                     │
                               Concatenate([decoder_outputs, attention_context])
                                     │
                               Dense(output_vocab_size, softmax)
                                     │
                               predicted next token
```

**Why cross-attention here?** The decoder uses the encoder output as both Key and Value (`K=enc, V=enc`) and its own output as Query (`Q=dec`). At each decoding step the model asks: *"which encoder positions are most relevant for generating the next target word?"*

> **Network routing analogy 🔄:** The attention weights act as a **dynamic routing table** — the decoder queries the encoder sequence and attention weights determine which encoder positions to route information from at each time step, like adaptive load balancing across source nodes.

In [ ]:
from tensorflow.keras.layers import Concatenate, Dense, Embedding, Input, LSTM
from tensorflow.keras.models import Model


def build_seq2seq(attention_class=None, optimizer='adam'):
    """🏗️ Build and compile the Encoder-Decoder + Attention model.

    Parameters
    ----------
    attention_class : Layer subclass
        SelfAttention variant to use. Defaults to Glorot version.
    optimizer : str
        Keras optimiser string ('adam', 'adagrad', etc.).
    """
    if attention_class is None:
        attention_class = SelfAttention

    # ── Encoder ──────────────────────────────────────────────────────────
    enc_inputs          = Input(shape=(max_input_length,))
    enc_emb             = Embedding(input_vocab_size, 256)(enc_inputs)
    enc_lstm            = LSTM(256, return_sequences=True, return_state=True)
    enc_outputs, h, c   = enc_lstm(enc_emb)
    enc_states          = [h, c]

    # ── Decoder ──────────────────────────────────────────────────────────
    dec_inputs          = Input(shape=(max_output_length - 1,))
    dec_emb             = Embedding(output_vocab_size, 256)(dec_inputs)
    dec_lstm            = LSTM(256, return_sequences=True, return_state=True)
    dec_outputs, _, _   = dec_lstm(dec_emb, initial_state=enc_states)

    # ── Cross-Attention: Q=decoder, K=encoder, V=encoder ─────────────────
    attn_output         = attention_class()([dec_outputs, enc_outputs, enc_outputs])

    # ── Fusion + output projection ────────────────────────────────────────
    fused               = Concatenate(axis=-1)([dec_outputs, attn_output])
    output              = Dense(output_vocab_size, activation='softmax')(fused)

    # ── Compile ───────────────────────────────────────────────────────────
    model = Model([enc_inputs, dec_inputs], output)
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


# Build baseline model
model_baseline = build_seq2seq()
model_baseline.summary()

---
## 📈 Part 5 — Training (Baseline: Glorot init + Adam)

I use **categorical cross-entropy** as the loss function:

$$\mathcal{L} = -\sum_{t=1}^{T} \sum_{v=1}^{V} y_{t,v} \log \hat{y}_{t,v}$$

where $y_{t,v}$ is the one-hot target and $\hat{y}_{t,v}$ is the predicted probability for vocabulary token $v$ at timestep $t$.

**Adam** maintains per-parameter adaptive learning rates using first and second gradient moment estimates — converging faster than plain SGD on dense sequence tasks. The Adam update rule is:

$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

where $\hat{m}_t$ and $\hat{v}_t$ are bias-corrected estimates of the first and second gradient moments.

In [ ]:
# 🔄 Train baseline model (Glorot + Adam)
history_glorot_adam = model_baseline.fit(
    [input_sequences, decoder_input_data],
    decoder_output_data,
    epochs=100,
    batch_size=16,
    verbose=1
)

In [ ]:
# 📊 Plot baseline training loss
plt.figure(figsize=(8, 4))
plt.plot(history_glorot_adam.history['loss'], color='steelblue', linewidth=2, label='Glorot + Adam (baseline)')
plt.title('📉 Training Loss — Baseline', fontsize=13)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Final loss: {history_glorot_adam.history["loss"][-1]:.4f}')

---
## 🧪 Part 6 — Experiment A: Glorot vs He Initialisation

I now retrain the same architecture using **He uniform** initialisation instead of Glorot, then compare the two loss curves.

### Weight initialisers compared 🔢

| Initialiser | Scale | Designed for |
|---|---|---|
| **Glorot uniform** | $\sqrt{\frac{6}{fan\_in + fan\_out}}$ | Symmetric activations (`tanh`, `sigmoid`, linear projections) |
| **He uniform** | $\sqrt{\frac{6}{fan\_in}}$ | One-sided activations (`ReLU`, `ELU`, `SELU`) |

**Why the difference?** ReLU zeroes the negative half of its input, halving the effective activation variance. He compensates by using a larger initial weight scale.

> **Power control analogy 📡:** Choosing an initialiser is like setting the initial transmit power in a power-control loop. Glorot calibrates for a symmetric channel; He calibrates for a half-wave rectifier receiver. Using He with linear LSTM projections is like slightly over-driving the transmitter — it may converge faster or overshoot depending on network depth.

In [ ]:
# 🏗️ SelfAttention with He uniform initialisation
class SelfAttentionHe(Layer):
    """Self-Attention with He uniform weight initialisation."""

    def __init__(self, **kwargs):
        super(SelfAttentionHe, self).__init__(**kwargs)

    def build(self, input_shape):
        feature_dim = input_shape[0][-1]
        self.Wq = self.add_weight(shape=(feature_dim, feature_dim), initializer='he_uniform', trainable=True, name='Wq')
        self.Wk = self.add_weight(shape=(feature_dim, feature_dim), initializer='he_uniform', trainable=True, name='Wk')
        self.Wv = self.add_weight(shape=(feature_dim, feature_dim), initializer='he_uniform', trainable=True, name='Wv')
        super(SelfAttentionHe, self).build(input_shape)

    def call(self, inputs):
        q, k, v = inputs
        q = K.dot(q, self.Wq)
        k = K.dot(k, self.Wk)
        v = K.dot(v, self.Wv)
        scores = K.batch_dot(q, k, axes=[2, 2])
        dk     = K.cast(K.shape(k)[-1], dtype=K.floatx())
        scores = scores / K.sqrt(dk)
        attention_weights = K.softmax(scores, axis=-1)
        return K.batch_dot(attention_weights, v)

    def compute_output_shape(self, input_shape):
        return input_shape[0]


print('✅ SelfAttentionHe defined')

In [ ]:
# 🔄 Train model with He uniform initialisation
model_he = build_seq2seq(attention_class=SelfAttentionHe, optimizer='adam')

history_he = model_he.fit(
    [input_sequences, decoder_input_data],
    decoder_output_data,
    epochs=100,
    batch_size=16,
    verbose=1
)

In [ ]:
# 📊 Compare Glorot vs He
plt.figure(figsize=(9, 4))
plt.plot(history_glorot_adam.history['loss'], label='Glorot uniform + Adam', color='red',  linewidth=2)
plt.plot(history_he.history['loss'],          label='He uniform + Adam',     color='blue', linewidth=2)
plt.title('📊 Training Loss — Glorot vs He Initialisation (Adam optimiser)', fontsize=12)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Final loss — Glorot + Adam : {history_glorot_adam.history["loss"][-1]:.4f}')
print(f'Final loss — He + Adam     : {history_he.history["loss"][-1]:.4f}')

**🔍 Interpretation:** On this tiny dataset both initialisers converge to a comparable final loss. At larger scale and depth, Glorot tends to outperform He in LSTM-dominated architectures (symmetric activations), while He wins in ReLU-heavy feed-forward networks. Most production Transformers use Glorot for attention projections and He for feed-forward sub-layers.

---
## 🧪 Part 7 — Experiment B: Adam vs Adagrad

I now swap the optimiser from Adam to **Adagrad** and compare the loss curves.

### Optimiser comparison 🔢

| Property | Adam | Adagrad |
|---|---|---|
| **Gradient history** | EWMA (exponential moving average) — old gradients fade | Cumulative sum $G_t = \sum_{\tau=1}^{t} g_\tau^2$ — grows forever |
| **Update rule** | $\theta \leftarrow \theta - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$ | $\theta \leftarrow \theta - \frac{\eta}{\sqrt{G_t + \epsilon}} g_t$ |
| **Risk** | Mild overshoot in late training | $G_t \to \infty$ → learning rate → 0, training stalls |
| **Best for** | Dense updates (sequences, images) | Sparse features (large embedding look-ups) |

> **Congestion control analogy 🔄:** Adam is like **CUBIC TCP** — it weights recent history more heavily so old congestion events fade out. Adagrad is like a TCP variant with no reset — every past collision permanently reduces throughput for that flow, which is conservative but can stall on long training runs.

In [ ]:
# 🔄 Train same architecture with Adagrad optimiser
model_adagrad = build_seq2seq(attention_class=SelfAttention, optimizer='adagrad')

history_adagrad = model_adagrad.fit(
    [input_sequences, decoder_input_data],
    decoder_output_data,
    epochs=100,
    batch_size=16,
    verbose=1
)

In [ ]:
# 📊 Compare Adam vs Adagrad
plt.figure(figsize=(9, 4))
plt.plot(history_glorot_adam.history['loss'], label='Glorot + Adam',    color='red',  linewidth=2)
plt.plot(history_adagrad.history['loss'],     label='Glorot + Adagrad', color='blue', linewidth=2)
plt.title('📊 Training Loss — Adam vs Adagrad (Glorot initialisation)', fontsize=12)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Final loss — Glorot + Adam    : {history_glorot_adam.history["loss"][-1]:.4f}')
print(f'Final loss — Glorot + Adagrad : {history_adagrad.history["loss"][-1]:.4f}')

**🔍 Interpretation:** Adam typically converges faster and to a lower loss on dense LSTM update tasks. Adagrad can plateau early because its accumulator $G_t$ only grows, shrinking the effective learning rate monotonically. For large sparse NLP problems (e.g. training word embeddings on sparse co-occurrence data) Adagrad has historically performed well — but Adam is the safer default for end-to-end neural translation.

---
## 📊 Summary

### What I built

| Component | Details |
|---|---|
| **Encoder** | `Embedding(256)` → `LSTM(256, return_sequences=True)` → context states $(h, c)$ |
| **Decoder** | `Embedding(256)` → `LSTM(256, init_state=enc_states)` → decoder outputs |
| **Attention** | `SelfAttention`: Q=decoder, K=encoder, V=encoder → attended context |
| **Fusion** | `Concatenate([decoder_outputs, attention_context])` |
| **Output** | `Dense(output_vocab_size, softmax)` → probability over vocabulary |
| **Training** | Teacher forcing — ground-truth previous token fed at each step |

### Experiment results

| Config | Initialiser | Optimiser | Verdict |
|---|---|---|---|
| ✅ Baseline | Glorot uniform | Adam | Best general-purpose choice |
| 🔵 Exp A | He uniform | Adam | Similar loss; He preferred for ReLU-heavy nets |
| 🔴 Exp B | Glorot uniform | Adagrad | Slower convergence — accumulator stalls learning rate |

### Key formulas

| Concept | Formula |
|---|---|
| Scaled dot-product attention | $\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$ |
| Glorot scale | $\sigma = \sqrt{\frac{6}{fan\_in + fan\_out}}$ |
| He scale | $\sigma = \sqrt{\frac{6}{fan\_in}}$ |
| Adam update | $\theta \leftarrow \theta - \frac{\eta}{\sqrt{\hat{v}_t}+\epsilon}\hat{m}_t$ |
| Categorical cross-entropy | $\mathcal{L} = -\sum_t \sum_v y_{t,v}\log\hat{y}_{t,v}$ |

### Connected concepts
[[self_attention]] · [[encoder_decoder]] · [[seq2seq]] · [[weight_initialisation]] · [[adam_optimizer]] · [[adagrad_optimizer]] · [[categorical_crossentropy]] · [[transformer_architecture]]

---
## 🧪 Sandbox — Try It Yourself

Use this section to experiment freely without breaking the main notebook.

In [ ]:
# 🧪 SANDBOX — experiment freely here

# Ideas to try:
# 1. Change embedding dimension from 256 to 128 — does it converge as well?
# 2. Try optimizer='rmsprop' — how does it compare to Adam?
# 3. Add a Dropout layer after the decoder LSTM
# 4. Increase to 200 epochs — does loss continue to drop?
# 5. Add more sentence pairs and see how vocab size changes
# 6. Replace SelfAttention with Keras's built-in MultiHeadAttention

# --- your code below ---


In [ ]:
# 🧪 Quick comparison helper — plot any two training histories
def compare_histories(h1, h2, label1='Model 1', label2='Model 2', title='Loss Comparison'):
    plt.figure(figsize=(9, 4))
    plt.plot(h1.history['loss'], label=label1, linewidth=2)
    plt.plot(h2.history['loss'], label=label2, linewidth=2)
    plt.title(title)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.tight_layout()
    plt.show()

# Usage:
# compare_histories(history_glorot_adam, history_he, 'Glorot+Adam', 'He+Adam')
